 BRONZE TO SILVER LAYER (Cleaning & Transformation) ,SILVER TO GOLD LAYER (Star Schema Modeling - Fact & Dimensions)

In [0]:
import io
import pandas as pd
from azure.storage.blob import ContainerClient

storage_account = "adlsproject06"
container_name = "superstore-sales-dataset"
access_key = os.getenv("AZURE_STORAGE_ACCESS_KEY")

# Azure Storage Connection String banayein
connection_string = f"DefaultEndpointsProtocol=https;AccountName={storage_account};AccountKey={access_key};EndpointSuffix=core.windows.net"
container_client = ContainerClient.from_connection_string(connection_string, container_name)

print("--- Step 1: Reading Bronze CSV from ADLS ---")
blob_client = container_client.get_blob_client("bronze/train.csv")
blob_data = blob_client.download_blob().readall()

# Memory stream se directly Pandas DataFrame me load karein (Encoding latin1 use ki hai taaki koi character error na aaye)
df_bronze = pd.read_csv(io.BytesIO(blob_data), encoding="latin1")
print(f"Bronze Data Loaded. Total Rows: {len(df_bronze)}")


# ==============================================================================
# STEP 2: BRONZE TO SILVER LAYER (Cleaning & Transformation)
# ==============================================================================
print("--- Step 2: Running Silver Transformations ---")

# Duplicates aur missing essential IDs ko drop karein
df_silver = df_bronze.drop_duplicates().dropna(subset=["Order ID", "Customer ID", "Product ID"])

# Text formatting
df_silver["Customer Name"] = df_silver["Customer Name"].str.strip()
df_silver["Segment"] = df_silver["Segment"].str.strip().str.upper()

# Cleaned data ko Silver layer me CSV format me upload karein
silver_csv_data = df_silver.to_csv(index=False)
container_client.get_blob_client("silver/sales_cleaned.csv").upload_blob(silver_csv_data, overwrite=True)
print("Silver layer cleaned data uploaded successfully to ADLS!")


# ==============================================================================
# STEP 3: SILVER TO GOLD LAYER (Star Schema Modeling - Fact & Dimensions)
# ==============================================================================
print("--- Step 3: Running Gold Star Schema Transformations ---")

# 1. Customer Dimension Table
dim_customer = df_silver[["Customer ID", "Customer Name", "Segment", "Country", "City", "State", "Postal Code", "Region"]].drop_duplicates()
container_client.get_blob_client("gold/dim_customer.csv").upload_blob(dim_customer.to_csv(index=False), overwrite=True)

# 2. Product Dimension Table
dim_product = df_silver[["Product ID", "Category", "Sub-Category", "Product Name"]].drop_duplicates()
container_client.get_blob_client("gold/dim_product.csv").upload_blob(dim_product.to_csv(index=False), overwrite=True)

# 3. Sales Fact Table
fact_sales = df_silver[["Row ID", "Order ID", "Order Date", "Ship Date", "Ship Mode", "Customer ID", "Product ID", "Sales"]]
container_client.get_blob_client("gold/fact_sales.csv").upload_blob(fact_sales.to_csv(index=False), overwrite=True)

print("Gold layer Star Schema tables (Fact & Dimensions) uploaded successfully to ADLS!")
print("Pipeline execution completed successfully!")

--- Step 1: Reading Bronze CSV from ADLS ---
Bronze Data Loaded. Total Rows: 9800
--- Step 2: Running Silver Transformations ---
Silver layer cleaned data uploaded successfully to ADLS!
--- Step 3: Running Gold Star Schema Transformations ---
Gold layer Star Schema tables (Fact & Dimensions) uploaded successfully to ADLS!
Pipeline execution completed successfully!


In [0]:
import io
import pandas as pd
from azure.storage.blob import ContainerClient

storage_account = "adlsproject06"
container_name = "superstore-sales-dataset"
access_key = os.getenv("AZURE_STORAGE_ACCESS_KEY")

connection_string = f"DefaultEndpointsProtocol=https;AccountName={storage_account};AccountKey={access_key};EndpointSuffix=core.windows.net"
container_client = ContainerClient.from_connection_string(connection_string, container_name)

# 1. Read Bronze Data
blob_client = container_client.get_blob_client("bronze/train.csv")
df_bronze = pd.read_csv(io.BytesIO(blob_client.download_blob().readall()), encoding="latin1")

# 2. Check Duplicates and Missing Values
print(f"Total Bronze Rows: {len(df_bronze)}")
print(f"Total Duplicate Rows: {df_bronze.duplicated().sum()}")
print("\nMissing Values Count per Column:")
print(df_bronze.isnull().sum())

# 3. Read Silver Data to see difference
blob_silver = container_client.get_blob_client("silver/sales_cleaned.csv")
df_silver = pd.read_csv(io.BytesIO(blob_silver.download_blob().readall()))
print(f"\nTotal Silver Rows after cleaning: {len(df_silver)}")
print(f"Total Rows Removed/Cleaned: {len(df_bronze) - len(df_silver)}")

Total Bronze Rows: 9800
Total Duplicate Rows: 0

Missing Values Count per Column:
Row ID            0
Order ID          0
Order Date        0
Ship Date         0
Ship Mode         0
Customer ID       0
Customer Name     0
Segment           0
Country           0
City              0
State             0
Postal Code      11
Region            0
Product ID        0
Category          0
Sub-Category      0
Product Name      0
Sales             0
dtype: int64

Total Silver Rows after cleaning: 9800
Total Rows Removed/Cleaned: 0
